# 12 — Full-cycle Gemma/Tunix actor + separate critic → PPO evaluation

Эта тетрадка убирает smoke learner из финального контура. Она строит CrafText runtime, MegaPrompts renderer, настоящий `GemmaTunixBackend`, собирает batched rollout, сохраняет replay, превращает его в `TextTrajectoryBatch`, затем отдельно вызывает actor scoring (`new_logprobs`, entropy) и critic scoring (values). На границе objective считается PPO loss и проверяются все ключевые значения.

Важно: notebook не скачивает веса и не использует mock fallback. Перед запуском положите локальный Gemma snapshot в `artifacts/models/gemma3-270m-it`.


In [ ]:
from pathlib import Path
import json
import time

import jax
import jax.numpy as jnp

from tunix.models.automodel import call_model_config

from tunix_craftext.batched_rollout import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.config import load_mvp_config
from tunix_craftext.llm_ppo import evaluate_separate_llm_actor_critic_ppo
from tunix_craftext.profiling import PhaseProfiler, save_profile
from tunix_craftext.prompts import MegaPromptRenderer
from tunix_craftext.runtime import build_craftext_runtime
from tunix_craftext.text_trajectory import text_trajectory_from_replay
from tunix_craftext.tunix_actor import (
    build_gemma_tunix_actor,
    init_linear_value_head,
    merge_actor_critic_scores,
)


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

CONFIG_PATH = ROOT / 'configs/mvp/tiny_craftext.yaml'
MODEL_PROFILE = ROOT / 'configs/models/gemma3_270m_instruction.yaml'
SNAPSHOT = ROOT / 'artifacts/models/gemma3-270m-it'
TRAJECTORY_DIR = ROOT / 'artifacts/trajectories/gemma-full-cycle-notebook'
PROFILE_PATH = ROOT / 'artifacts/profiles/gemma-full-cycle-notebook.json'

BATCH_SIZE = 2
HORIZON = 2
MAX_NEW_TOKENS = 12
CACHE_SIZE = 1024
SEED = 17

if not SNAPSHOT.is_dir():
    raise FileNotFoundError(
        f'Gemma snapshot is required and must be local: {SNAPSHOT}. '
        'The notebook intentionally does not download model assets.'
    )


## 1. Build CrafText runtime, MegaPrompts and Gemma actor/critic

`GemmaTunixActor` owns generation and actor token scoring. `actor.critic()` returns a separate critic role over the same Tunix backbone plus value head. Later this boundary maps naturally to `RLCluster` actor/critic role meshes.


In [ ]:
config = load_mvp_config(CONFIG_PATH)
runtime = build_craftext_runtime(config)
renderer = MegaPromptRenderer(config.prompt.template)

gemma_config = call_model_config('gemma3_270m_it')
value_head = init_linear_value_head(jax.random.PRNGKey(SEED), hidden_dim=gemma_config.embed_dim)
actor = build_gemma_tunix_actor(
    profile_path=MODEL_PROFILE,
    snapshot=SNAPSHOT,
    cache_size=CACHE_SIZE,
    value_head=value_head,
    seed=SEED,
)
critic = actor.critic()

print('runtime actions:', runtime.action_count)
print('model:', actor.profile.model_id, 'hidden_dim:', gemma_config.embed_dim)


## 2. Collect a real batched rollout and export replay evidence

The model sees MegaPrompts output, Tunix generates text/tokens/logprobs, strict decode maps the response to a legal action, and CrafText advances via the adapter.


In [ ]:
profiler = PhaseProfiler()

with profiler.section('rollout'):
    rollout = collect_batched_text_rollout(
        runtime.adapter,
        renderer,
        actor.backend,
        actions=runtime.actions,
        batch_size=BATCH_SIZE,
        horizon=HORIZON,
        seed=SEED,
        goal='Play safely and make progress toward the current scenario instruction.',
        max_new_tokens=MAX_NEW_TOKENS,
        invalid_action='fallback',
        fallback_action_id=0,
    )

replays = replays_from_batched_rollout(
    rollout,
    config_path=str(CONFIG_PATH.relative_to(ROOT)),
    commit='notebook-local',
    backend='tunix-single-device:Gemma',
)
TRAJECTORY_DIR.mkdir(parents=True, exist_ok=True)
for index, replay in enumerate(replays):
    path = TRAJECTORY_DIR / f'env-{index}.json'
    path.write_text(json.dumps(replay.to_json(), ensure_ascii=False, indent=2), encoding='utf-8')

print('decisions:', len(rollout.decisions), 'envs:', len(replays))
print('saved:', TRAJECTORY_DIR)
print('first completion:', rollout.decisions[0].responses[0].raw_text[:220])


## 3. Replay → token batch

Each replay step becomes one token-learning row. Rewards are placed on the final generated token, fallback rows remain auditable but are excluded from `policy_mask`.


In [ ]:
batch = text_trajectory_from_replay(replays[0])
batch.validate()

print('token_ids:', batch.token_ids.shape)
print('prompt_token_ids:', batch.prompt_token_ids.shape)
print('valid generated tokens:', int(jnp.sum(batch.token_mask)))
print('policy tokens:', int(jnp.sum(batch.policy_mask)))
print('fallback rows:', batch.fallback_used.tolist())


## 4. Actor scoring + separate critic scoring

Actor recomputes `new_logprobs` and entropy for generated tokens. Critic independently recomputes token values. They are merged only at the PPO objective boundary.


In [ ]:
with profiler.section('actor_scoring'):
    actor_scores = actor.score_actor_tokens(
        prompt_token_ids=batch.prompt_token_ids,
        prompt_token_mask=batch.prompt_token_mask,
        token_ids=batch.token_ids,
        token_mask=batch.token_mask,
    )

with profiler.section('critic_scoring'):
    critic_values = critic.score_values(
        prompt_token_ids=batch.prompt_token_ids,
        prompt_token_mask=batch.prompt_token_mask,
        token_ids=batch.token_ids,
        token_mask=batch.token_mask,
    )

scores = merge_actor_critic_scores(actor_scores, critic_values, batch.token_ids)

assert bool(jnp.all(jnp.isfinite(actor_scores.token_logprobs[batch.token_mask])))
assert bool(jnp.all(jnp.isfinite(actor_scores.entropy[batch.token_mask])))
assert bool(jnp.all(jnp.isfinite(critic_values.values[batch.token_mask])))

print('old_logprobs:', batch.old_logprobs.tolist())
print('new_logprobs:', actor_scores.token_logprobs.tolist())
print('critic values:', critic_values.values.tolist())
print('entropy:', actor_scores.entropy.tolist())


## 5. PPO evaluation over real actor/critic values

This cell does not update parameters yet. It verifies the exact values that a full Optax update consumes: returns, advantages, log-prob ratio path, value loss and entropy.


In [ ]:
with profiler.section('ppo_evaluation'):
    evaluation = evaluate_separate_llm_actor_critic_ppo(
        batch,
        actor_scores,
        critic_values,
        learning_mask=batch.policy_mask,
        gamma=0.99,
        clip_epsilon=0.2,
        value_coefficient=0.5,
        entropy_coefficient=0.01,
    )

evaluation.validate(batch)
assert bool(jnp.isfinite(evaluation.loss))
assert bool(jnp.all(jnp.isfinite(evaluation.returns[batch.token_mask])))
assert bool(jnp.all(jnp.isfinite(evaluation.advantages[batch.token_mask])))

print('returns:', evaluation.returns.tolist())
print('advantages:', evaluation.advantages.tolist())
print('loss:', float(evaluation.loss))
print({name: float(value) for name, value in evaluation.metrics.items()})


## 6. Save phase profile

The saved JSON is intentionally compatible with the repository dashboard/profiling lane. Formal accelerator profiling can wrap the same sections with NVIDIA JAX-Toolbox / `nsys-jax`.


In [ ]:
save_profile(
    PROFILE_PATH,
    profiler.events(),
    metadata={
        'notebook': '12_full_cycle_craftext_training.ipynb',
        'model': actor.profile.model_id,
        'batch_size': BATCH_SIZE,
        'horizon': HORIZON,
        'max_new_tokens': MAX_NEW_TOKENS,
    },
)
print('profile saved:', PROFILE_PATH)
for name, event in profiler.summary().items():
    print(name, event)
